# Testing file 
### where we evaluate Zhang's models using the test set

## Preliminaries

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from tensorflow.keras.optimizers import Adam
from tensorflow.data import Dataset


from util.load_data import load_data
from util.evaluation import *
from models.zhang.models import FairLogisticRegression
from models.zhang.learning import train_loop as zhang_train

/Users/lffpl/Projects/falsb/env/falsb/lib/python3.11/site-packages/keras/src/export/tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


In [2]:
batch_size = 64
epochs = 100
lr = 0.001

In [3]:
cv_seeds = [13, 29, 42, 55, 73]

## Load data

In [4]:
data_name = 'heart-age'

In [5]:
x, y, a = load_data(data_name)
raw_data = (x, y, a)

In [6]:
xdim = x.shape[1]
ydim = y.shape[1]
adim = a.shape[1]
zdim = 8

In [7]:
print(xdim, ydim, adim, zdim)

25 1 1 8


In [8]:
print(len(x))

1025


## Result file

In [9]:
header = "model_name", "cv_seed", "clas_acc", "dp", "deqodds", "deqopp", "trade_dp", "trade_deqodds", "trade_deqopp", "TN_a0", "FP_a0", "FN_a0", "TP_a0", "TN_a1", "FP_a1", "FN_a1", "TP_a1"
results = []

## Testing loop
#### Each model is evalueted 5 times
#### In the end of each iteration we save the result

### Zhang for DP

In [10]:
fairdef = 'DemPar'

for cv_seed in cv_seeds:
    x_train, x_test, y_train, y_test, a_train, a_test = train_test_split(
        x, y, a, test_size=0.3, random_state=cv_seed)

    train_data = Dataset.from_tensor_slices((x_train, y_train, a_train))
    train_data = train_data.batch(batch_size, drop_remainder=True)

    test_data = Dataset.from_tensor_slices((x_test, y_test, a_test))
    test_data = test_data.batch(batch_size, drop_remainder=True)

    # train below

    opt = Adam(learning_rate=lr)

    model = FairLogisticRegression(xdim, ydim, adim, batch_size, fairdef)
    zhang_train(model, raw_data, train_data, epochs, opt)

    Y, A, Y_hat, A_hat = fair_evaluation(model, test_data)
    clas_acc, dp, deqodds, deqopp, confusion_matrix, metrics_a0, metrics_a1 = compute_metrics(Y, A, Y_hat, A_hat, adim)

    fair_metrics = (dp, deqodds, deqopp)
    tradeoff = []
    for fair_metric in fair_metrics:
        tradeoff.append(compute_tradeoff(clas_acc, fair_metric))

    result = ['Zhang4DP', cv_seed, clas_acc, dp, deqodds, deqopp, tradeoff[0], tradeoff[1], tradeoff[2]] + metrics_a0 + metrics_a1

    results.append(result)

    del(opt)

> Epoch | Class Loss | Adv Loss | Class Acc | Adv Acc


2026-01-05 15:19:25.660014: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


> 1 | 0.7784185409545898 | 1.0366919040679932 | 0.5142045454545454 | 0.2911931818181818


2026-01-05 15:19:25.922995: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


> 2 | 0.7597723007202148 | 1.0259597301483154 | 0.5142045454545454 | 0.2911931818181818
> 3 | 0.7429815530776978 | 1.0189704895019531 | 0.5142045454545454 | 0.2911931818181818


...

> 64 | 0.34868282079696655 | 0.7585574388504028 | 0.8650568181818182 | 0.3991477272727273
> 65 | 0.34768053889274597 | 0.7553073763847351 | 0.8650568181818182 | 0.4005681818181818
> 66 | 0.3467041254043579 | 0.7520856261253357 | 0.8622159090909091 | 0.4161931818181818
> 67 | 0.34580081701278687 | 0.7488693594932556 | 0.8622159090909091 | 0.4275568181818182
> 68 | 0.344927579164505 | 0.7456879615783691 | 0.8622159090909091 | 0.4303977272727273
> 69 | 0.34408047795295715 | 0.7425422668457031 | 0.8622159090909091 | 0.4346590909090909
> 70 | 0.34325891733169556 | 0.7394322156906128 | 0.8636363636363636 | 0.4403409090909091
> 71 | 0.34246134757995605 | 0.736355721950531 | 0.8650568181818182 | 0.4502840909090909
> 72 | 0.34166181087493896 | 0.7332993745803833 | 0.8650568181818182 | 0.4616477272727273
> 73 | 0.34086763858795166 | 0.7302431464195251 | 0.8650568181818182 | 0.46448863636363635
> 74 | 0.34010380506515503 | 0.7271987199783325 | 0.8650568181818182 | 0.46448863636363635
> 75 | 0.339

2026-01-05 15:20:01.452132: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


> 27 | 0.4567466378211975 | 0.8901451230049133 | 0.859375 | 0.3039772727272727
> 28 | 0.4484265148639679 | 0.8873130679130554 | 0.8622159090909091 | 0.3039772727272727
> 29 | 0.4405544400215149 | 0.8844327926635742 | 0.8622159090909091 | 0.3039772727272727
> 30 | 0.43318426609039307 | 0.8815503120422363 | 0.8636363636363636 | 0.3039772727272727
> 31 | 0.426052451133728 | 0.8786975145339966 | 0.8664772727272727 | 0.3039772727272727
> 32 | 0.4191857576370239 | 0.8758265972137451 | 0.8664772727272727 | 0.3039772727272727
> 33 | 0.41267839074134827 | 0.872938871383667 | 0.8650568181818182 | 0.3039772727272727
> 34 | 0.40637844800949097 | 0.870078444480896 | 0.8622159090909091 | 0.3039772727272727
> 35 | 0.40035080909729004 | 0.8672688603401184 | 0.859375 | 0.3039772727272727
> 36 | 0.39479586482048035 | 0.8645057082176208 | 0.859375 | 0.3039772727272727
> 37 | 0.38943079113960266 | 0.8617632389068604 | 0.8607954545454546 | 0.3039772727272727
> 38 | 0.3843374252319336 | 0.8589882254600525 |

2026-01-05 15:20:43.169476: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


> 54 | 0.4575589895248413 | 0.8088492155075073 | 0.8636363636363636 | 0.3380681818181818
> 55 | 0.4562094807624817 | 0.806091845035553 | 0.8636363636363636 | 0.3366477272727273
> 56 | 0.4549444913864136 | 0.8033915758132935 | 0.8650568181818182 | 0.3465909090909091
> 57 | 0.4537341594696045 | 0.8007208704948425 | 0.8650568181818182 | 0.3508522727272727
> 58 | 0.45259904861450195 | 0.7981456518173218 | 0.8650568181818182 | 0.3565340909090909
> 59 | 0.45157307386398315 | 0.7955864667892456 | 0.8650568181818182 | 0.3678977272727273
> 60 | 0.45069968700408936 | 0.7930227518081665 | 0.8650568181818182 | 0.3721590909090909
> 61 | 0.4498712122440338 | 0.7904942631721497 | 0.8622159090909091 | 0.37642045454545453
> 62 | 0.44910669326782227 | 0.787982702255249 | 0.8636363636363636 | 0.3821022727272727
> 63 | 0.4484143853187561 | 0.7854254245758057 | 0.8636363636363636 | 0.38636363636363635
> 64 | 0.44779735803604126 | 0.7829102277755737 | 0.8636363636363636 | 0.390625
> 65 | 0.4472690224647522 

### Zhang for Eq Odds

In [11]:
fairdef = 'EqOdds'

for cv_seed in cv_seeds:
    x_train, x_test, y_train, y_test, a_train, a_test = train_test_split(
        x, y, a, test_size=0.3, random_state=cv_seed)

    train_data = Dataset.from_tensor_slices((x_train, y_train, a_train))
    train_data = train_data.batch(batch_size, drop_remainder=True)

    test_data = Dataset.from_tensor_slices((x_test, y_test, a_test))
    test_data = test_data.batch(batch_size, drop_remainder=True)

    # train below

    opt = Adam(learning_rate=lr)
    
    model = FairLogisticRegression(xdim, ydim, adim, batch_size, fairdef)
    zhang_train(model, raw_data, train_data, epochs, opt)

    Y, A, Y_hat, A_hat = fair_evaluation(model, test_data)
    clas_acc, dp, deqodds, deqopp, confusion_matrix, metrics_a0, metrics_a1 = compute_metrics(Y, A, Y_hat, A_hat, adim)

    fair_metrics = (dp, deqodds, deqopp)
    tradeoff = []
    for fair_metric in fair_metrics:
        tradeoff.append(compute_tradeoff(clas_acc, fair_metric))

    result = ['Zhang4EqOdds', cv_seed, clas_acc, dp, deqodds, deqopp, tradeoff[0], tradeoff[1], tradeoff[2]] + metrics_a0 + metrics_a1

    results.append(result)

    del(opt)

> Epoch | Class Loss | Adv Loss | Class Acc | Adv Acc
> 1 | 0.7784185409545898 | 1.032285213470459 | 0.5142045454545454 | 0.2911931818181818
> 2 | 0.7597721815109253 | 1.0173568725585938 | 0.5142045454545454 | 0.2911931818181818
> 3 | 0.7429815530776978 | 1.006317138671875 | 0.5142045454545454 | 0.2911931818181818
> 4 | 0.7267346382141113 | 0.995612621307373 | 0.5142045454545454 | 0.2911931818181818
> 5 | 0.7109712958335876 | 0.9852118492126465 | 0.5142045454545454 | 0.2911931818181818
> 6 | 0.6956647634506226 | 0.9751686453819275 | 0.5142045454545454 | 0.2911931818181818


2026-01-05 15:22:19.724787: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


> 7 | 0.6808714866638184 | 0.9654911756515503 | 0.5142045454545454 | 0.2911931818181818
> 8 | 0.6666208505630493 | 0.9561760425567627 | 0.5142045454545454 | 0.2911931818181818
> 9 | 0.6529024243354797 | 0.9472150802612305 | 0.5142045454545454 | 0.2911931818181818
> 10 | 0.6395975947380066 | 0.9386488199234009 | 0.5142045454545454 | 0.2911931818181818
> 11 | 0.62668776512146 | 0.9304476380348206 | 0.515625 | 0.2911931818181818
> 12 | 0.6142222285270691 | 0.9226741790771484 | 0.5426136363636364 | 0.2911931818181818
> 13 | 0.6022228002548218 | 0.9152923822402954 | 0.5767045454545454 | 0.2911931818181818
> 14 | 0.590632975101471 | 0.9082393646240234 | 0.6207386363636364 | 0.2911931818181818
> 15 | 0.5793828964233398 | 0.9015393853187561 | 0.6633522727272727 | 0.2911931818181818
> 16 | 0.568526029586792 | 0.8951908946037292 | 0.7045454545454546 | 0.2911931818181818
> 17 | 0.5580607652664185 | 0.8891710638999939 | 0.7414772727272727 | 0.2911931818181818
> 18 | 0.547981858253479 | 0.883420586

### Zhang for Eq Opp

In [12]:
fairdef = 'EqOpp'

for cv_seed in cv_seeds:
    x_train, x_test, y_train, y_test, a_train, a_test = train_test_split(
        x, y, a, test_size=0.3, random_state=cv_seed)

    train_data = Dataset.from_tensor_slices((x_train, y_train, a_train))
    train_data = train_data.batch(batch_size, drop_remainder=True)

    test_data = Dataset.from_tensor_slices((x_test, y_test, a_test))
    test_data = test_data.batch(batch_size, drop_remainder=True)

    # train below

    opt = Adam(learning_rate=lr)
    
    model = FairLogisticRegression(xdim, ydim, adim, batch_size, fairdef)
    zhang_train(model, raw_data, train_data, epochs, opt)

    Y, A, Y_hat, A_hat = fair_evaluation(model, test_data)
    clas_acc, dp, deqodds, deqopp, confusion_matrix, metrics_a0, metrics_a1 = compute_metrics(Y, A, Y_hat, A_hat, adim)

    fair_metrics = (dp, deqodds, deqopp)
    tradeoff = []
    for fair_metric in fair_metrics:
        tradeoff.append(compute_tradeoff(clas_acc, fair_metric))

    result = ['Zhang4EqOpp', cv_seed, clas_acc, dp, deqodds, deqopp, tradeoff[0], tradeoff[1], tradeoff[2]] + metrics_a0 + metrics_a1

    results.append(result)

    del(opt)

> Epoch | Class Loss | Adv Loss | Class Acc | Adv Acc


> 1 | 0.77911376953125 | 0.49935489892959595 | 0.5142045454545454 | 0.2911931818181818
> 2 | 0.7611020803451538 | 0.4935581088066101 | 0.5142045454545454 | 0.2911931818181818
> 3 | 0.7448993921279907 | 0.4895641803741455 | 0.5142045454545454 | 0.2911931818181818
> 4 | 0.7291048765182495 | 0.4856377840042114 | 0.5142045454545454 | 0.2911931818181818
> 5 | 0.7138487100601196 | 0.4817650616168976 | 0.5142045454545454 | 0.2911931818181818
> 6 | 0.6990161538124084 | 0.477935254573822 | 0.5142045454545454 | 0.2911931818181818
> 7 | 0.684597909450531 | 0.47416430711746216 | 0.5142045454545454 | 0.2911931818181818
> 8 | 0.6707528829574585 | 0.47043997049331665 | 0.5142045454545454 | 0.2911931818181818
> 9 | 0.6574023962020874 | 0.46676135063171387 | 0.5142045454545454 | 0.2911931818181818
> 10 | 0.6444363594055176 | 0.4631250500679016 | 0.5142045454545454 | 0.2911931818181818
> 11 | 0.6318726539611816 | 0.4595341682434082 | 0.515625 | 0.2911931818181818
> 12 | 0.6196318864822388 | 0.4559811353

2026-01-05 15:26:25.458199: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


> 14 | 0.5964217782020569 | 0.44903773069381714 | 0.59375 | 0.2911931818181818
> 15 | 0.5853621363639832 | 0.44563087821006775 | 0.6335227272727273 | 0.2911931818181818
> 16 | 0.5746780633926392 | 0.4422801733016968 | 0.671875 | 0.2911931818181818
> 17 | 0.5643512010574341 | 0.43898871541023254 | 0.7102272727272727 | 0.2911931818181818
> 18 | 0.5543209314346313 | 0.43574026226997375 | 0.7372159090909091 | 0.2911931818181818
> 19 | 0.5446405410766602 | 0.43254393339157104 | 0.7698863636363636 | 0.2911931818181818
> 20 | 0.5352996587753296 | 0.42940840125083923 | 0.7897727272727273 | 0.2911931818181818
> 21 | 0.5262951254844666 | 0.4263344705104828 | 0.8039772727272727 | 0.2911931818181818
> 22 | 0.5176694393157959 | 0.4232971668243408 | 0.8167613636363636 | 0.2911931818181818
> 23 | 0.5093072652816772 | 0.42027339339256287 | 0.828125 | 0.2911931818181818
> 24 | 0.5012980699539185 | 0.4172850251197815 | 0.8380681818181818 | 0.2911931818181818
> 25 | 0.4935528635978699 | 0.414365381002426

## Saving into DF then CSV

In [13]:
result_df = pd.DataFrame(results, columns=header)
result_df

,model_name,cv_seed,clas_acc,dp,deqodds,deqopp,trade_dp,trade_deqodds,trade_deqopp,TN_a0,FP_a0,FN_a0,TP_a0,TN_a1,FP_a1,FN_a1,TP_a1
0,Zhang4DP,13,0.859375,0.742011,0.934176,0.890688,0.796392,0.895216,0.874752,20.0,4.0,4.0,53.0,83.0,14.0,14.0,64.0
1,Zhang4DP,29,0.875000,0.740603,0.956903,0.927470,0.802211,0.914121,0.900471,16.0,3.0,3.0,47.0,89.0,15.0,11.0,72.0
2,Zhang4DP,42,0.832031,0.694383,0.881212,0.837425,0.757001,0.855916,0.834719,14.0,2.0,2.0,45.0,92.0,23.0,16.0,62.0
3,Zhang4DP,55,0.855469,0.824458,0.876421,0.900163,0.839677,0.865818,0.877247,26.0,2.0,3.0,56.0,75.0,21.0,11.0,62.0
4,Zhang4DP,73,0.851562,0.760663,0.973252,0.980561,0.803550,0.908350,0.911520,15.0,4.0,6.0,50.0,84.0,18.0,10.0,69.0
5,Zhang4EqOdds,13,0.859375,0.742011,0.934176,0.890688,0.796392,0.895216,0.874752,20.0,4.0,4.0,53.0,83.0,14.0,14.0,64.0
6,Zhang4EqOdds,29,0.878906,0.755096,0.944251,0.927470,0.812310,0.910408,0.902535,17.0,2.0,3.0,47.0,89.0,15.0,11.0,72.0
7,Zhang4EqOdds,42,0.828125,0.699564,0.876865,0.837425,0.758435,0.851798,0.832749,14.0,2.0,2.0,45.0,91.0,24.0,16.0,62.0
8,Zhang4EqOdds,55,0.855469,0.824458,0.876421,0.900163,0.839677,0.865818,0.877247,26.0,2.0,3.0,56.0,75.0,21.0,11.0,62.0
9,Zhang4EqOdds,73,0.851562,0.760663,0.973252,0.980561,0.803550,0.908350,0.911520,15.0,4.0,6.0,50.0,84.0,18.0,10.0,69.0


In [14]:
result_df.to_csv(f'{data_name}-result/zhang-{epochs}.csv')